# Airbnb európai szálláshelyek elemzése

Ebben a notebookban mesterséges intelligencia eszközök segítségével elemzünk Airbnb szállásadatokat – adatbetöltéstől a klaszterezésen át az ár-előrejelzésig.

### Mit fogunk csinálni?
1. Betöltjük és megvizsgáljuk a 10 európai várost tartalmazó adatbázist
2. **Gyors EDA** – az adatok vizualizálása az első lépésben
3. Feltárjuk az összefüggéseket korreláció-heatmap és boxplot diagramokkal
4. Log-transzformációval kezeljük az ár eloszlásának ferdeségét
5. **K-means klaszterezéssel** (tanítás nélküli AI) szegmentáljuk a budapesti szálláshelyeket
6. **Random Forest** modellel előrejelezzük az árakat
7. Cosine similarity alapján hasonló szállásokat ajánlunk *(opcionális)*

### Eszközök
- **pandas, numpy** – adatkezelés
- **matplotlib, seaborn** – vizualizáció
- **scikit-learn** – klaszterezés, gépi tanulás
- **Gemini** – kódgenerálás (vibe coding)

> **Vibe coding:** nem mi írjuk a kódot, hanem AI-jal generáltatjuk. Mi pedig értjük, futtatjuk és irányítjuk.

## 1. Telepítés és importok

Az első lépés a szükséges csomagok betöltése. Ezt a cellát mindig futtasd le először!

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

print('Csomagok sikeresen betöltve!')

Csomagok sikeresen betöltve!


## 2. Adatbetöltés

Az adatfájlt töltsd fel a Colab session-be (bal oldali fájlkezelő → feltöltés gomb),
vagy add meg a GitHub raw URL-t.

**A dataset oszlopai:**

| Oszlop | Tartalom |
|--------|----------|
| City | Város (10 európai főváros) |
| Price | Éjszakánkénti ár (EUR) |
| Day | Weekday / Weekend |
| Room Type | Private room / Entire home / Shared room |
| Superhost | Szupergazda-e? |
| Person Capacity | Max. vendégszám |
| Cleanliness Rating | Tisztasági értékelés |
| Guest Satisfaction | Összesített értékelés (0–100) |
| Bedrooms | Hálószobák száma |
| City Center (km) | Távolság a városközponttól |
| Metro Distance (km) | Távolság a legközelebbi metróhoz |
| Normalised Attraction Index | Látványosságok indexe (0–100) |

---
**Prompt feladat**

Fogalmazz meg egy promptot a Gemini számára, amely betölti az adatfájlt és kiírja az alapinformációkat!

Kontextus:
- A fájl neve: `airbnb_europe.csv`
- Írja ki a sorok és oszlopok számát
- Jelenítse meg az első 5 sort
- Írja ki az összes város nevét
- Ellenőrizze a hiányzó értékeket

*A kapott kódot másold be az alábbi cellába és futtasd le!*

In [1]:
# IDE KERÜL A GEMINI ÁLTAL GENERÁLT KÓD
# Adatbetöltés és alapinformációk



## 2.5 Gyors EDA – az adatok első pillantásra

Mielőtt bármit szűrünk vagy transzformálunk, nézzük meg az adatbázist vizuálisan.
Ez a két diagram megmutatja, **miért** lesz szükség outlier szűrésre és log-transzformációra.

**Amit keresünk:**
- Melyik városban drágábbak a szállások?
- Milyen az ár eloszlása? Szimmetrikus, vagy van néhány extrém magas érték?

---
**Prompt feladat**

Fogalmazz meg egy promptot a Gemini számára, amely elkészíti az EDA vizualizációkat!

Kontextus:
- `df` a betöltött DataFrame
- **1. diagram:** barplot – átlagos `Price` városonként, csökkenő sorrendben, `palette='Set3'`, méret 10×5
- **2. diagram:** hisztogram – `Price` nyers eloszlása, `bins=60`, méret 10×4

*A kapott kódot másold be az alábbi cellába és futtasd le!*

In [2]:
# IDE KERÜL A GEMINI ÁLTAL GENERÁLT KÓD
# Gyors EDA: átlagárak városonként + ár hisztogram



## 3. Alapstatisztikák és outlier szűrés

Az ár eloszlása erősen jobbra ferde – néhány extrém árú luxusszálláshely torzítja az elemzést.
A 99. percentilis feletti rekordokat eltávolítjuk.

---
**Prompt feladat**

Fogalmazz meg egy promptot a Gemini számára, amely elvégzi az alapstatisztikákat és az outlier szűrést!

Kontextus:
- `df` a betöltött DataFrame
- Írja ki a `describe()` eredményét
- Számítsa ki a 99. percentilis értéket az árra
- Szűrje ki az e feletti sorokat
- Írja ki, hány sor maradt és mi a szűrési határ (EUR/éj)

*A kapott kódot másold be az alábbi cellába és futtasd le!*

In [3]:
# IDE KERÜL A GEMINI ÁLTAL GENERÁLT KÓD
# Alapstatisztikák és outlier szűrés




## 4. Korreláció-heatmap

A heatmap megmutatja, melyik változók függnek össze egymással.
Az értékek –1 és +1 között mozognak:
- **+1** → erős pozitív összefüggés (ha az egyik nő, a másik is nő)
- **–1** → erős negatív összefüggés (ha az egyik nő, a másik csökken)
- **0** → nincs összefüggés

---
**Prompt feladat**

Fogalmazz meg egy promptot a Gemini számára, amely elkészíti a korreláció-heatmapet!

Kontextus:
- `df` a szűrt DataFrame
- Csak numerikus oszlopokat szerepeltess
- Seaborn heatmap, `annot=True`, `cmap='RdBu_r'`, `vmin=-1`, `vmax=1`
- Méret: 14×9

*A kapott kódot másold be az alábbi cellába és futtasd le!*

In [4]:
# IDE KERÜL A GEMINI ÁLTAL GENERÁLT KÓD
# Korreláció-heatmap



## 5. Ár eloszlás városonként – log-transzformáció

A nyers ár eloszlása miatt a KDE grafikonon a városok alig különböztethetők meg.
A **log10-transzformáció** összenyomja a nagy értékeket és szétnyújtja a kis értékeket,
így az eloszlások összehasonlíthatóvá válnak.

Például: 100 EUR → 2.0, 1000 EUR → 3.0, 10 000 EUR → 4.0

---
**Prompt feladat**

Fogalmazz meg egy promptot a Gemini számára, amely elkészíti a log-transzformált ár KDE diagramot!

Kontextus:
- `df` a szűrt DataFrame
- Hozz létre egy `df_log` másolatot, ahol `Price` = `np.log10(df['Price'])`
- `sns.displot` KDE diagram, `hue='City'`, `fill=True`
- Cím: `'Ár eloszlás városonként (log10 skála)'`

*A kapott kódot másold be az alábbi cellába és futtasd le!*

In [5]:
# IDE KERÜL A GEMINI ÁLTAL GENERÁLT KÓD
# Log-transzformált ár KDE diagram városonként



## 6. Összefüggés-vizsgálatok (binning)

Három kérdést vizsgálunk boxplot diagramokkal:
1. Befolyásolja-e a **tisztaság** a vendégelégedettséget?
2. Csökken-e az **ár** a városközponttól távolodva?
3. Magasabb ár jár-e együtt jobb **értékeléssel**?

A **binning** technika folytonos változókat (pl. távolság km-ben) kategóriákra oszt
(pl. 0–3 km, 3–6 km stb.), így boxplottal vizualizálhatók.

---
**Prompt feladat**

Fogalmazz meg egy promptot a Gemini számára, amely elkészíti a három boxplot diagramot!

Kontextus:
- `df` a szűrt DataFrame, `df_log` a log-transzformált verzió
- **1. diagram:** `x='Cleanliness Rating'`, `y='Guest Satisfaction'`
- **2. diagram:** `City Center (km)` binning 3 km-es lépésekkel (0–3, 3–6, … 24–27 km),
  `x=bin`, `y=log10(Price)`
- **3. diagram:** `Guest Satisfaction` binning 10-es lépésekkel (0–10, 10–20, … 90–100),
  `x=bin`, `y=log10(Price)`
- Minden diagramnál `palette='Set3'`, méret: 12×5

*A kapott kódot másold be az alábbi cellába és futtasd le!*

In [6]:
# IDE KERÜL A GEMINI ÁLTAL GENERÁLT KÓD
# Összefüggés-vizsgálatok: cleanliness, távolság, elégedettség



## 7. K-means klaszterezés – Budapest

A K-means tanítás nélküli (unsupervised) algoritmus: előre megcímkézett adat nélkül fedez fel csoportokat a hasonlóságok alapján. Egyszerre veszi figyelembe az összes változót – ár, távolság, elégedettség, hálószobák száma stb. – így üzleti szegmensek azonosíthatók (pl. budget, prémium, nagy kapacitású). A folytonos változókat standardizálni kell, mert eltérő skálán vannak; a kategorikus oszlopokat one-hot encodinggal bináris (0/1) értékekké alakítjuk.

**Lépések:**
1. Szűrjük Budapest adatait
2. Kategóriaváltozókat kódolunk (one-hot encoding)
3. Folytonos változókat standardizálunk (`StandardScaler`)
4. Elbow method – meghatározzuk az optimális K értéket
5. K-means futtatása, klaszter profilok
6. PCA vizualizáció – 2D-ben ábrázoljuk a klasztereket

> **A PCA két tengelye nem közvetlen változók** – a variancia irányait mutatják.
> Az a lényeg, hogy a **színek szétválnak-e**: ez jelzi, hogy az algoritmus valódi csoportokat talált.

---
**Prompt feladat – 1. rész: adatelőkészítés és elbow**

Fogalmazz meg egy promptot a Gemini számára, amely elvégzi a Budapest szűrést,
az adatelőkészítést és az elbow method kirajzolását!

Kontextus:
- `df` a teljes szűrt DataFrame
- Szűrj Budapest adataira: `df[df['City'] == 'Budapest']`
- One-hot encoding a kategorikus oszlopokra (`pd.get_dummies`), bool → int8 konverzió
- `StandardScaler` a folytonos numerikus oszlopokra
- Elbow method: K = 2–15, inertia görbe

*A kapott kódot másold be az alábbi cellába és futtasd le!*

In [7]:
# IDE KERÜL A GEMINI ÁLTAL GENERÁLT KÓD
# Budapest szűrés, adatelőkészítés, elbow method



---
**Prompt feladat – 2. rész: K-means és PCA vizualizáció**

Az elbow görbe alapján válaszd ki az optimális K értéket, majd fogalmazz meg
egy promptot a Gemini számára a klaszterezés befejezéséhez!

Kontextus:
- `X_scaled` a standardizált mátrix, `df_bp` a budapesti DataFrame
- `KMeans(n_clusters=K, n_init='auto', random_state=42)`
- Klaszter profilok: csoportonkénti átlagok a főbb változókra
  (Price, Guest Satisfaction, City Center (km), Bedrooms)
- PCA 2 komponensre, scatter plot klaszterszín szerint

*A kapott kódot másold be az alábbi cellába és futtasd le!*

In [8]:
# IDE KERÜL A GEMINI ÁLTAL GENERÁLT KÓD
# K-means futtatás, klaszter profilok, PCA vizualizáció



## 8. Ár-előrejelzés – Random Forest (Budapest)

A **Random Forest** egy döntési fák sokaságából álló ensemble modell.
Most arra tanítjuk, hogy a szálláshely jellemzői alapján megjósolja az árat.

**Értékelési metrikák:**
- **MAE** (Mean Absolute Error) – átlagos abszolút hiba EUR-ban
- **RMSE** (Root Mean Squared Error) – a nagy hibákat jobban bünteti
- **R²** – mennyit magyaráz meg a modell a variabilitásból (0–1, magasabb a jobb)

**Feature importance:** megmutatja, melyik változó befolyásolja leginkább az árat.

---
**Prompt feladat**

Fogalmazz meg egy promptot a Gemini számára, amely felépíti és értékeli a Random Forest modellt!

Kontextus:
- `df_bp` a budapesti DataFrame (one-hot kódolt, az előző feladatból)
- Cél változó: `Price`
- Train/test split: 80/20, `random_state=42`
- `RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)`
- Írja ki a Train és Test MAE, RMSE, R² értékeket
- Készítsen top 15 feature importance barplot diagramot (vízszintes)

*A kapott kódot másold be az alábbi cellába és futtasd le!*

In [9]:
# IDE KERÜL A GEMINI ÁLTAL GENERÁLT KÓD
# Random Forest ár-előrejelzés, metrikák, feature importance



## 9. Hasonló szállások ajánlója *(opcionális)*

> Ez a feladat opcionális – ha az idő engedi, vagy házi projektként.

A **cosine similarity** két vektor közötti szög koszinuszát méri.
Ha két szálláshely numerikus jellemzői hasonlók (ár, értékelés, elhelyezkedés stb.),
a köztük lévő szög kicsi → a koszinusz értéke 1-hez közelít.

Az ajánló logikája:
1. Minden szálláshelyet egy numerikus vektor reprezentál
2. A referencia szálláshely vektorát összehasonlítjuk az összes többivel
3. A legmagasabb hasonlósági értékű 5 szállást ajánljuk

**Továbbfejlesztési irány – embedding-alapú ajánló:**
A mostani megközelítés numerikus jellemzőkön dolgozik.
A modern ajánlórendszerek szöveges leírásokból nyelvi modell segítségével
**beágyazási vektorokat (embeddings)** generálnak, és szemantikai hasonlóságot mérnek.
Ehhez a `sentence-transformers` könyvtár használható Colabon, API kulcs nélkül
(`paraphrase-multilingual-MiniLM` modell) – ez a házi projekt egyik lehetséges iránya.

---
**Prompt feladat**

Fogalmazz meg egy promptot a Gemini számára, amely felépíti a cosine similarity ajánlót!

Kontextus:
- `df` a teljes szűrt DataFrame (mind a 10 város)
- Feature-ök: `Price`, `Guest Satisfaction`, `City Center (km)`,
  `Bedrooms`, `Normalised Attraction Index`, `Metro Distance (km)`
- Standardizáld a feature-öket `StandardScaler`-rel
- Írj egy `ajanlj(idx, top_n=5)` függvényt, amely kiírja a referencia szállást
  és a top 5 leghasonlóbbat (Város, Room Type, Ár, Km, Értékelés, Hasonlóság)
- Teszteld egy budapesti és egy párizsi szállásra

*A kapott kódot másold be az alábbi cellába és futtasd le!*

In [10]:
# IDE KERÜL A GEMINI ÁLTAL GENERÁLT KÓD
# Cosine similarity ajánló

